# Program for running and testing a baseline fasttext model

Organization of notebook:

- [Setup import of pacakges](#setup)
- [Read and clean data](#data)
- [Fit fasttext model](#fit)
- [Predict and calculate performance results](#eval)
- [Save predictions and results to temporary folder](#save)

<a id="setup"></a>
## Setup import of packages 

In [ ]:
import pandas as pd
import os
from datetime import datetime
import time
import sys
from pathlib import Path
import json

import fasttext as ft
from sklearn.metrics import f1_score, accuracy_score

from utils.fasttext_process import cleaning_df, predict_and_evaluate
from utils.formatting import reduce_test, get_sn
from config import train_path, test_path

In [ ]:
temp_data_path = "temp" # Change to SSPcloud?

<a id="filter"></a>
## Read and clean data

In [ ]:
train = pd.read_parquet(train_path)
test = pd.read_parquet(test_path)

In [ ]:
# Remove 00.000 in training
train = train.loc[train.nace_21_code != "00.000",:]
print(f'Training data has shape {train.shape}')

# Reduce test - As processing and modelling takes a long time we are using a subset
test = test.reset_index(drop=True)
test = reduce_test(test)
print(f'Test data has shape {test.shape}')

In [ ]:
# Clean and preprocess texts
train = cleaning_df(train)
test = cleaning_df(test)

# Create one string
train['tekst_fasttext'] = train.apply(
    lambda row: f"__label__{row['nace_21_code']} {row['company_name']} {row['company_purpose']} {row['company_activity']}", 
    axis=1
)
test['tekst_fasttext'] = test.apply(
    lambda row: f"{row['company_name']} {row['company_purpose']} {row['company_activity']}", 
    axis=1
)

test_text = test['tekst_fasttext'].tolist()

<a id="fit"></a>
## Fit fasttext model 

In [ ]:
# Save training to files for fasttext
train['tekst_fasttext'].to_csv(f'{temp_data_path}/train_fasttext.txt', index=False, header=False)

# Start timing
t_start = time.time()

# Train model - hyperparameters from Mina
model_best = ft.train_supervised(input=f'{temp_data_path}/train_fasttext.txt', 
                                 lr=0.149, 
                                 epoch=18,
                                 wordNgrams=2,
                                 dim=100,
                                 loss="softmax")

<a id="eval"></a>
## Predict and calculate performance results

In [ ]:
# top 1 results
labels_1, probs_1 = model_best.predict(test_text)

# top 5 results
labels_5, probs_5 = model_best.predict(test_text, k=5)

In [ ]:
# Evaluation of model
results = fattext_evaulate(test, test, labels_1, probs_1, labels_5, probs_5)

<a id="save"></a>
## Save predictions and results to temporary folder (change to SSPCloud)

In [ ]:
# Save as JSON for easy reloading
with open(temp_data_path / f'top5_predictions_small.json', 'w') as f:
    json.dump(pred_labels_5, f)

In [ ]:
# Save model performance results
date_str = datetime.now().strftime("%Y-%m-%d")
filename = f"src/results/results_fasttext_{date_str}.txt"

with open(filename, "w") as f:
        f.write(f"Classified testdata of size {test.shape[0]} \n")
        f.write(f"Valid percentage: {results[valid_pct)}\n")
        f.write(f"Accuracy: {results[exact_pct)}\n")
        f.write(f"Accuracy (4.digits): {results[exact4_pct)}\n")
        f.write(f"F1-macro: {results[f1_macro)}\n")
        f.write(f"F1-weighted: {results[f1_weighted)}\n")
        f.write(f"Top 5 accuracy: {results[top5_accuracy)}\n")
        f.write(f"Run time: {round(results[total_time), 2)} minutes\n")